# Walkthrough: forecasting India's CPI inflation

This notebook walks through the analysis step by step, calling the same functions as `scripts/run_all.py`.
Run `python scripts/run_all.py` first so the processed panel and backtest results exist.

In [ ]:
import sys; sys.path.insert(0, "../src")
import pandas as pd, matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from inflation_fc import config as C, stationarity, regression, var_model
from inflation_fc.data import load_panel, load_meta
from inflation_fc.sarima import SarimaForecaster, select_order
pd.set_option("display.precision", 3)
panel = load_panel(); meta = load_meta()
print("Data:", meta.get("data"), "|", meta.get("start"), "to", meta.get("end"))
panel[["cpi", "inflation", "repo", "iip_yoy", "oil_yoy", "inr_yoy"]].tail()

## 1. Look at the data

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.axhspan(2, 6, alpha=0.1); ax.axhline(4, ls="--", lw=1)
panel[["inflation", "repo"]].plot(ax=ax); ax.set_title("CPI inflation and repo rate");

## 2. Stationarity
ADF (H0: unit root) and KPSS (H0: stationary) together. Note the verdict for log CPI vs its difference.

In [ ]:
st = stationarity.stationarity_table(panel)
st[["variable", "adf_p", "kpss_p", "za_break", "verdict"]]

## 3. Identify a SARIMA model
ACF/PACF of the differenced log CPI (regular + seasonal difference) guide the search.

In [ ]:
dd = panel["log_cpi"].diff().diff(12).dropna()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
plot_acf(dd, lags=36, ax=axes[0]); plot_pacf(dd, lags=36, ax=axes[1], method="ywm");

In [ ]:
pre = panel.loc[: pd.Timestamp(C.BACKTEST_START) - pd.offsets.MonthBegin(1)]
grid = select_order(pre["log_cpi"])
grid.head(8)

In [ ]:
best = grid.iloc[0]
sarima = SarimaForecaster(best["order"], best["seasonal_order"])
res = sarima.fit_full(panel)
print(res.summary())

In [ ]:
res.plot_diagnostics(figsize=(11, 7));

## 4. Phillips curve and Taylor rule (Newey-West errors)

In [ ]:
pc = regression.phillips_curve(panel)
print(pc.summary_text)
pc.diagnostics, pc.chow

In [ ]:
regression.subsample_comparison(panel, regression.phillips_curve)

In [ ]:
tr = regression.taylor_rule(panel)
{k: v for k, v in tr.extra.items() if not isinstance(v, pd.Series)}

## 5. VAR: Granger causality and impulse responses

In [ ]:
print(var_model.lag_selection(pre).round(3))
vres = var_model.fit_var(panel)
print("lags:", vres.k_ar, "| stable:", vres.is_stable())
var_model.granger_table(vres).round(3)

In [ ]:
vres.irf(24).plot(orth=True, impulse="repo", figsize=(10, 9));

In [ ]:
var_model.fevd_table(vres).round(1)

## 6. Out-of-sample comparison
Produced by `run_all.py` (rolling-origin backtest).

In [ ]:
met = pd.read_csv(C.RESULTS / "metrics.csv")
met.pivot(index="model", columns="h", values="rmse").sort_values(3)

In [ ]:
pd.read_csv(C.RESULTS / "dm_vs_random_walk.csv")

## 7. Take-aways
- Log CPI has a unit root and its first difference is stationary, which motivates d = 1 in the SARIMA model.
- In the rolling backtest SARIMA is the most accurate model at 1, 3 and 12 months; tree-based ML models beat the random walk but not SARIMA on ~140 monthly observations.
- The estimated Taylor rule satisfies the Taylor principle (long-run inflation response above 1).
- VAR impulse responses to a repo shock are imprecisely estimated; treating oil as exogenous (VARX) is a natural next step.